# Texture Robustness: ResNet-50 và ViT-B/16

**Câu hỏi nghiên cứu.** Khi các chi tiết texture bị suy giảm, prediction
của hai model thay đổi ra sao? Notebook sử dụng đúng manifest ImageNet
10.000 ảnh của nhóm, Gaussian Blur và Bilateral Filtering cố định. Nó
đánh giá *sensitivity to texture-detail degradation*, không coi blur là
một phép đo texture bias thuần túy.

In [1]:
from pathlib import Path
import sys, json

ROOT = Path.cwd()
if not (ROOT / 'src').is_dir(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display

from src.data import iter_analysis_images, load_analysis_manifest
from src.models import load_model
from experiments.texture.run_texture_robustness import (
    CONDITIONS, apply_transform, baseline_rows, evaluate_condition, seed_everything, summarise,
)
plt.style.use('seaborn-v0_8-whitegrid')
OUT = ROOT / 'results' / 'tv4'; FIGURES = OUT / 'figures'; FIGURES.mkdir(parents=True, exist_ok=True)
DATA_DIR = ROOT / 'data' / 'imagenet' / 'data'
expected = [f'validation-{i:05d}-of-00014.parquet' for i in range(14)]
DATA_AVAILABLE = [p.name for p in sorted(DATA_DIR.glob('validation-*.parquet'))] == expected
seed_everything(42)
print('Thiết bị inference:', 'CUDA — ' + torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

Thiết bị inference: CUDA — NVIDIA GeForce RTX 3060


## 1. Trực giác và protocol

Gaussian Blur làm yếu nhanh các chi tiết tần số cao và biên mịn. Bilateral
Filtering làm mịn trong vùng màu tương tự nhưng cố gắng giữ biên. Mỗi phép
biến đổi được áp dụng trên RGB gốc trước official preprocessing, giữ nguyên
kích thước và ground-truth label. Severity 1→3 tăng dần cường độ.

In [2]:
parameter_rows = []
for transformation, severities in CONDITIONS.items():
    for severity, parameters in severities.items():
        parameter_rows.append({'Transformation': transformation, 'Severity': severity, **parameters})
pd.DataFrame(parameter_rows)

,Transformation,Severity,kernel_size,sigma,diameter,sigma_color,sigma_space
0,gaussian_blur,1,5.0,1.0,NaN,NaN,NaN
1,gaussian_blur,2,9.0,2.0,NaN,NaN,NaN
2,gaussian_blur,3,15.0,4.0,NaN,NaN,NaN
3,bilateral_filter,1,NaN,NaN,5.0,25.0,25.0
4,bilateral_filter,2,NaN,NaN,9.0,50.0,50.0
5,bilateral_filter,3,NaN,NaN,13.0,100.0,100.0


## 2. Dataset và model setup

Manifest cố định chứa đúng 10 ảnh/class từ ImageNet validation. Cell kiểm
tra file trước để không âm thầm thay dataset. Model luôn lấy checkpoint và
transform từ loader chung.

In [3]:
manifest = load_analysis_manifest()
print(f'Manifest: {len(manifest):,} ảnh, {len({r["label"] for r in manifest}):,} classes, 10 ảnh/class.')
display(pd.DataFrame(manifest[:5]))
if DATA_AVAILABLE:
    display(Markdown(f'✅ Đủ 14 validation shards tại `{DATA_DIR}`.'))
else:
    display(Markdown(
        f'⚠️ **Không có 14 ImageNet validation shards tại `{DATA_DIR}`.** '
        'Dataset nguồn ILSVRC/imagenet-1k là gated và lượt tải chính thức không xác thực trả về 401. '
        'Các cell cần pixel sẽ tự bỏ qua; không có metric texture nào được tạo.'))

Manifest: 10,000 ảnh, 1,000 classes, 10 ảnh/class.


,dataset_index,image_id,label
0,36674,ILSVRC2012_val_00000005_n03125729.JPEG,516
1,16489,ILSVRC2012_val_00000011_n01917289.JPEG,109
2,28069,ILSVRC2012_val_00000014_n04065272.JPEG,757
3,759,ILSVRC2012_val_00000015_n03496892.JPEG,595
4,44073,ILSVRC2012_val_00000029_n01729977.JPEG,55


⚠️ **Không có 14 ImageNet validation shards tại `/home/honganh/Documents/tempt/cnn-vit-interpretability/data/imagenet/data`.** Dataset nguồn ILSVRC/imagenet-1k là gated và lượt tải chính thức không xác thực trả về 401. Các cell cần pixel sẽ tự bỏ qua; không có metric texture nào được tạo.

In [4]:
resnet, resnet_transform, resnet_meta = load_model('resnet50')
vit, vit_transform, vit_meta = load_model('vit_b_16')
pd.DataFrame([
    {'Model': 'ResNet-50', 'Checkpoint': resnet_meta['weights'], 'Resize': resnet_meta['preprocessing']['resize_size'], 'Device': resnet_meta['device']},
    {'Model': 'ViT-B/16', 'Checkpoint': vit_meta['weights'], 'Resize': vit_meta['preprocessing']['resize_size'], 'Device': vit_meta['device']},
])

,Model,Checkpoint,Resize,Device
0,ResNet-50,ResNet50_Weights.IMAGENET1K_V2,[232],cuda
1,ViT-B/16,ViT_B_16_Weights.IMAGENET1K_V1,[256],cuda


## 3. Visualizing texture degradation

Nếu raw ImageNet có mặt, cell này lấy 4 IDs đầu manifest, áp dụng trực
tiếp đủ 6 transformations và hiển thị original/corrupted side-by-side.
Đây là visual demo trước mọi đánh giá định lượng.

In [5]:
sample_images = []
if DATA_AVAILABLE:
    sample_images = list(iter_analysis_images(manifest[:4], data_dir=DATA_DIR))
    columns = [('clean', 0)] + [(t, s) for t in CONDITIONS for s in (1, 2, 3)]
    fig, axes = plt.subplots(len(sample_images), len(columns), figsize=(17, 2.8 * len(sample_images)))
    for row_index, (image, label, image_id) in enumerate(sample_images):
        for column_index, (transformation, severity) in enumerate(columns):
            ax = axes[row_index, column_index]
            shown = image if transformation == 'clean' else apply_transform(image, transformation, severity)
            ax.imshow(shown); ax.axis('off')
            if row_index == 0: ax.set_title('Original' if transformation == 'clean' else f'{transformation}\nS{severity}', fontsize=8)
            if column_index == 0: ax.set_ylabel(f'ID: {image_id[:18]}\nlabel={label}', fontsize=7)
    fig.suptitle('Original và suy giảm texture trước preprocessing', y=1.01); fig.tight_layout()
    fig.savefig(FIGURES / 'texture_transformation_grid.png', dpi=300, bbox_inches='tight'); plt.show()
else:
    display(Markdown('⏸️ Grid không thể tạo: raw ImageNet pixels chưa có cục bộ.'))

⏸️ Grid không thể tạo: raw ImageNet pixels chưa có cục bộ.

## 4. Single-image model behavior

Code dưới chạy prediction trực tiếp trên original, Gaussian Blur nhẹ và
Gaussian Blur mạnh của cùng ảnh. Nó hiển thị top-1 ImageNet class,
confidence softmax và correctness cho từng model/condition; không dùng
prediction dựng sẵn.

In [6]:
from torchvision.models import ResNet50_Weights
imagenet_names = ResNet50_Weights.IMAGENET1K_V2.meta['categories']

def single_prediction(model, transform, image, true_label):
    device = next(model.parameters()).device
    tensor = transform(image).unsqueeze(0).to(device)
    with torch.inference_mode(), torch.autocast(device_type=device.type, enabled=device.type == 'cuda'):
        probabilities = torch.softmax(model(tensor).float(), dim=1)[0]
    confidence, prediction = probabilities.max(dim=0)
    return int(prediction), float(confidence), int(prediction) == int(true_label)

if DATA_AVAILABLE:
    image, label, image_id = sample_images[0]
    single_rows = []
    for condition, shown in [('Original', image), ('Gaussian S1', apply_transform(image, 'gaussian_blur', 1)), ('Gaussian S3', apply_transform(image, 'gaussian_blur', 3))]:
        for name, model, transform in [('ResNet-50', resnet, resnet_transform), ('ViT-B/16', vit, vit_transform)]:
            prediction, confidence, correct = single_prediction(model, transform, shown, label)
            single_rows.append({'Condition': condition, 'Model': name, 'Top-1': imagenet_names[prediction], 'Confidence': confidence, 'Correct': correct})
    display(Markdown(f'**Image ID:** `{image_id}`; **ground truth index:** `{label}`'))
    pd.DataFrame(single_rows).style.format({'Confidence': '{:.3f}'})
else:
    display(Markdown('⏸️ Single-image prediction cần ảnh ImageNet gốc nên được bỏ qua an toàn.'))

⏸️ Single-image prediction cần ảnh ImageNet gốc nên được bỏ qua an toàn.

## 5. Full Robustness Evaluation và prediction cache

Cache chỉ hợp lệ khi có đủ 2 model × (clean + 6 corruptions) × 10.000 IDs.
Nếu cache thiếu và raw data có mặt, cell gọi trực tiếp utility batch inference
cho từng condition trên cùng manifest; không yêu cầu chạy CLI riêng. Accuracy
drop được tính theo **percentage points** so với clean predictions cùng IDs.

In [7]:
texture_path = OUT / 'texture_predictions.csv'
summary_path = OUT / 'texture_summary.csv'
expected_rows = 2 * (1 + 6) * len(manifest)
cache_ok = texture_path.is_file() and len(pd.read_csv(texture_path)) == expected_rows
if cache_ok:
    texture_predictions = pd.read_csv(texture_path)
    texture_summary = pd.read_csv(summary_path)
    print('Đã dùng texture prediction cache hợp lệ.')
elif DATA_AVAILABLE:
    generated, summaries = [], []
    for model_name, model, transform in [('resnet50', resnet, resnet_transform), ('vit_b_16', vit, vit_transform)]:
        clean = baseline_rows(model_name, manifest); generated.extend(clean)
        for transformation in CONDITIONS:
            for severity in (1, 2, 3):
                changed = evaluate_condition(model, transform, model_name, transformation, severity, manifest, DATA_DIR, batch_size=64)
                generated.extend(changed)
                summaries.append({'model': model_name, 'transformation': transformation, 'severity': severity, **summarise(clean, changed, 42 + severity)})
    texture_predictions, texture_summary = pd.DataFrame(generated), pd.DataFrame(summaries)
    texture_predictions.to_csv(texture_path, index=False); texture_summary.to_csv(summary_path, index=False)
    print('Đã chạy full inference trực tiếp trong notebook và tạo cache.')
else:
    texture_predictions = texture_summary = None
    display(Markdown('⏸️ Không chạy full evaluation: thiếu 14 raw shards. Không có số liệu texture được suy diễn từ baseline.'))

⏸️ Không chạy full evaluation: thiếu 14 raw shards. Không có số liệu texture được suy diễn từ baseline.

In [8]:
if texture_summary is not None:
    display(texture_summary.sort_values(['transformation', 'severity', 'model']).style.format({
        'clean_top1_accuracy_percent': '{:.2f}', 'top1_accuracy_percent': '{:.2f}',
        'absolute_accuracy_drop_pp': '{:.2f}', 'prediction_consistency_percent': '{:.2f}'}))
    names, colors = {'resnet50': 'ResNet-50', 'vit_b_16': 'ViT-B/16'}, {'resnet50': '#377eb8', 'vit_b_16': '#e41a1c'}
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True)
    for ax, transformation in zip(axes, CONDITIONS):
        for model in names:
            rows = texture_summary[(texture_summary.model == model) & (texture_summary.transformation == transformation)].sort_values('severity')
            ax.plot(rows.severity, rows.top1_accuracy_percent, marker='o', label=names[model], color=colors[model])
        ax.set_title(transformation.replace('_', ' ').title()); ax.set_xlabel('Severity level'); ax.set_xticks([1, 2, 3]); ax.grid(alpha=.25)
    axes[0].set_ylabel('Top-1 accuracy (%)'); axes[1].legend(); fig.suptitle('Accuracy theo severity'); fig.tight_layout()
    fig.savefig(FIGURES / 'texture_accuracy_by_severity.png', dpi=300); plt.show()
else:
    display(Markdown('Không có biểu đồ accuracy/severity khi full evaluation chưa chạy.'))

Không có biểu đồ accuracy/severity khi full evaluation chưa chạy.

## 6. Qualitative failure analysis và discussion

Khi prediction cache tồn tại, section này lọc bằng code bốn nhóm: ResNet
đúng/ViT sai, ViT đúng/ResNet sai, cả hai sai, cả hai đúng ở severity mạnh;
sau đó hiển thị original/corrupted cùng prediction. Trong workspace hiện
tại, raw ImageNet và cache texture chưa có, nên không có case study nào
được giả lập.

**Trạng thái khoa học hiện tại:** Shape-vs-Texture Bias đã chạy full,
nhưng Texture Robustness chưa thể kết luận model nào robust hơn. Khi có
14 shards, Run All sẽ thực hiện demo, inference, metrics và figures theo
đúng cùng manifest. Blur làm thay đổi nhiều hơn texture, vì thế mọi diễn
giải sau này cần giữ cụm từ “texture-detail degradation”.

**References.** Shared protocol `docs/experiment_protocol.md`; ImageNet
validation source `ILSVRC/imagenet-1k`; torchvision weights trong
`src/models.py`.